# 第 7 周练习：代码评审严重级别分类器（Code Review Severity Classifier）

## 练习目标（理念）

构建一条 **QLoRA 微调流水线**，把代码评审（code review）评论自动分成严重级别。

**应用场景：** 帮团队给评审意见排优先级——自动判定为 **Critical / Major / Minor / Suggestion**。

## 和第 7 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 第 1 天：QLoRA 基础 | 量化（quantization）+ LoRA 适配器思路；本笔记本先做数据与基线 |
| 第 2 天：提示数据与基座模型 | `format_for_training`、Llama chat template、模型选型 |
| 第 3–4 天：GPU 上 QLoRA 训练 | 导出 JSONL / HuggingFace，在 Colab T4 上训练 |
| 第 5 天：评估与对比 | 零样本基线准确率 vs 微调后准确率 |

## 流水线步骤

1. **生成数据集** — 合成代码评审评论 + 严重级别标签
2. **基线评估** — 前沿模型零样本（zero-shot）表现
3. **准备训练数据** — 打成 QLoRA 可用的 chat 模板格式
4. **导出到 HuggingFace** — 供 Colab 训练拉取
5. **Gradio UI** — 交互式分类与评估界面

## 怎么跑

1. 准备 `.env`：优先 `OPENROUTER_API_KEY`；否则用默认 OpenAI 客户端环境变量
2. 从上到下运行单元格；先看数据集与划分，再跑基线预测
3. 需要 GPU 微调时：导出 JSONL / 推 Hub，把文末 Colab 代码拷到 T4 运行时

---

## 严重级别（Severity Levels）

| 级别 | 含义 | 例子 |
|------|------|------|
| **Critical** | 安全漏洞、数据丢失、崩溃风险 | SQL 注入、空指针、内存泄漏 |
| **Major** | Bug、功能坏掉、性能问题 | 逻辑错误、缺校验、N+1 查询 |
| **Minor** | 代码异味、可维护性 | 魔法数字、过长函数、缺文档 |
| **Suggestion** | 风格偏好、可选改进 | 命名、格式、重构建议 |


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 导入标准库 json：读写 JSON / JSONL 训练样本
import json
# 导入标准库 random：打乱数据集，保证可复现划分
import random
# 导入标准库 re：从模型回复里剥掉 ```json 代码围栏
import re
# 从 collections 导入 Counter：统计各严重级别出现次数
from collections import Counter
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端：调用 OpenRouter 或 OpenAI 的 Chat Completions
from openai import OpenAI
# 导入 gradio：搭交互式分类 UI
import gradio as gr

# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)


In [ ]:
# ========== 配置：模型与客户端（优先 OpenRouter，否则直连 OpenAI）==========

# OpenRouter 的 OpenAI 兼容网关地址
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
# 前沿小模型 id（OpenRouter 命名空间）；字符串勿改译，会影响实际调用的模型
FRONTIER_MODEL = "openai/gpt-4.1-mini"

# 先读 OPENROUTER_API_KEY；有则走 OpenRouter，没有则回退到默认 OpenAI
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    # base_url 指向 OpenRouter；api_key 用 OpenRouter 的 key
    client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)
    print("Using OpenRouter API")
else:
    # 无 OpenRouter key：用默认客户端（读 OPENAI_API_KEY 等）
    client = OpenAI()
    # 直连 OpenAI 时模型名不带 openai/ 前缀
    FRONTIER_MODEL = "gpt-4.1-mini"
    print("Using OpenAI API directly")


## 数据集：代码评审评论

合成（synthetic）代码评审反馈，每条带一个严重级别标签。后面用它做划分、基线评估和 QLoRA 训练格式化。


In [ ]:
# ========== 标签空间 + 合成数据集（comment 文本是训练/评估输入，保持英文）==========

# 四个严重级别：顺序固定，后面解析与统计都依赖这些字符串
SEVERITIES = ("Critical", "Major", "Minor", "Suggestion")

# 合成评审样本：每条 dict 含 comment（英文评审意见）与 severity（标签）
REVIEWS = [
    # --- Critical：安全 / 崩溃 / 数据丢失类（最高优先级）---
    {"comment": "This SQL query is vulnerable to injection. User input is concatenated directly into the query string without parameterization.", "severity": "Critical"},
    {"comment": "The password is being logged in plaintext here. This is a serious security violation that could expose user credentials.", "severity": "Critical"},
    {"comment": "No null check before dereferencing this pointer. This will cause a segfault when the object is None.", "severity": "Critical"},
    {"comment": "API key is hardcoded in the source code. This should be moved to environment variables immediately.", "severity": "Critical"},
    {"comment": "This DELETE endpoint has no authentication check. Anyone can delete any record without authorization.", "severity": "Critical"},
    {"comment": "Race condition in the transaction handler. Two concurrent requests could withdraw more than the available balance.", "severity": "Critical"},
    {"comment": "XSS vulnerability: user input is rendered directly in HTML without escaping. Attackers could inject malicious scripts.", "severity": "Critical"},
    {"comment": "The encryption key is derived from a predictable value. This makes the encryption trivially breakable.", "severity": "Critical"},
    {"comment": "Buffer overflow risk: memcpy with user-controlled size parameter and no bounds checking.", "severity": "Critical"},
    {"comment": "Session tokens are stored in localStorage which is vulnerable to XSS. Use httpOnly cookies instead.", "severity": "Critical"},
    {"comment": "This recursive function has no base case termination. It will cause stack overflow on any input.", "severity": "Critical"},
    {"comment": "Database connection is never closed in the finally block. This will exhaust the connection pool under load.", "severity": "Critical"},
    {"comment": "CSRF protection is disabled for this form. Attackers can forge requests on behalf of authenticated users.", "severity": "Critical"},
    {"comment": "Private key file has 777 permissions. This exposes cryptographic material to all users on the system.", "severity": "Critical"},
    {"comment": "Integer overflow possible when calculating buffer size. Could lead to heap corruption.", "severity": "Critical"},
    
    # --- Major：明确 Bug、功能损坏、性能问题 ---
    {"comment": "The comparison uses = instead of == which will always evaluate to true and assign rather than compare.", "severity": "Major"},
    {"comment": "This loop fetches each item individually causing N+1 query problem. Should use batch fetch or JOIN.", "severity": "Major"},
    {"comment": "Return value is ignored here. The function returns an error code that should be checked.", "severity": "Major"},
    {"comment": "Off-by-one error in the loop boundary. The last element will never be processed.", "severity": "Major"},
    {"comment": "The cache has no expiration policy. Stale data will be served indefinitely after the first request.", "severity": "Major"},
    {"comment": "Exception is caught but error is swallowed silently. Add logging or re-throw to avoid hiding failures.", "severity": "Major"},
    {"comment": "Missing input validation. Negative values will cause incorrect calculations in the price computation.", "severity": "Major"},
    {"comment": "This regex will cause catastrophic backtracking on certain inputs, leading to ReDoS vulnerability.", "severity": "Major"},
    {"comment": "The else branch returns undefined instead of an empty array, breaking the contract with callers.", "severity": "Major"},
    {"comment": "Async operation without await. The subsequent code assumes the operation completed but it hasn't.", "severity": "Major"},
    {"comment": "State mutation inside render method. This will cause infinite re-render loop in React.", "severity": "Major"},
    {"comment": "The timeout is set to 0ms which effectively disables the retry mechanism entirely.", "severity": "Major"},
    {"comment": "Default case is missing in the switch statement. Unexpected values will fall through silently.", "severity": "Major"},
    {"comment": "The file handle is opened but never closed if an exception occurs. Use context manager or try-finally.", "severity": "Major"},
    {"comment": "Date parsing assumes UTC but user input is in local timezone. This will cause off-by-hours bugs.", "severity": "Major"},
    {"comment": "Floating point comparison using == will fail for calculated values due to precision issues.", "severity": "Major"},
    
    # --- Minor：代码异味、可维护性 ---
    {"comment": "Magic number 86400 should be extracted to a named constant like SECONDS_PER_DAY for clarity.", "severity": "Minor"},
    {"comment": "This function is 200 lines long. Consider breaking it into smaller, focused helper functions.", "severity": "Minor"},
    {"comment": "The variable name 'x' is not descriptive. Rename to something meaningful like 'userCount' or 'index'.", "severity": "Minor"},
    {"comment": "Duplicate code block appears in 3 places. Extract to a shared utility function to reduce repetition.", "severity": "Minor"},
    {"comment": "This class has 15 dependencies injected. Consider splitting into smaller classes with single responsibility.", "severity": "Minor"},
    {"comment": "The boolean parameter makes call sites unclear. Consider using named arguments or separate methods.", "severity": "Minor"},
    {"comment": "Dead code: this else branch can never be reached based on the preceding conditions.", "severity": "Minor"},
    {"comment": "Missing docstring for this public API function. Add documentation explaining parameters and return value.", "severity": "Minor"},
    {"comment": "Cyclomatic complexity is 25. Refactor to reduce nesting and improve testability.", "severity": "Minor"},
    {"comment": "The TODO comment here is 2 years old. Either address it or remove if no longer relevant.", "severity": "Minor"},
    {"comment": "Inconsistent error message format. Other endpoints return {error: msg} but this returns {message: msg}.", "severity": "Minor"},
    {"comment": "This import is unused. Remove to keep the imports section clean and reduce bundle size.", "severity": "Minor"},
    {"comment": "Hard-coded string 'production' should use an enum or constant for environment names.", "severity": "Minor"},
    {"comment": "The test file is missing. Add unit tests for this new functionality before merging.", "severity": "Minor"},
    {"comment": "Type annotation is missing for the return value. Add -> Optional[User] for better IDE support.", "severity": "Minor"},
    {"comment": "Deeply nested callbacks here. Consider refactoring to async/await for better readability.", "severity": "Minor"},
    
    # --- Suggestion：风格与可选改进 ---
    {"comment": "Consider using f-strings instead of .format() for better readability in Python 3.6+.", "severity": "Suggestion"},
    {"comment": "This could be simplified to a list comprehension: [x.id for x in items if x.active]", "severity": "Suggestion"},
    {"comment": "The variable name follows camelCase but the codebase uses snake_case. Consider renaming for consistency.", "severity": "Suggestion"},
    {"comment": "Might be cleaner to use early return here instead of wrapping everything in an if block.", "severity": "Suggestion"},
    {"comment": "Optional: you could use dataclasses here to reduce boilerplate in the __init__ method.", "severity": "Suggestion"},
    {"comment": "The comments explain what the code does but not why. Consider adding context about the business logic.", "severity": "Suggestion"},
    {"comment": "This utility function might be useful in other modules. Consider moving to a shared helpers file.", "severity": "Suggestion"},
    {"comment": "Nit: extra blank line here doesn't match the style in the rest of the file.", "severity": "Suggestion"},
    {"comment": "The ternary expression is getting long. Might be more readable as a simple if-else block.", "severity": "Suggestion"},
    {"comment": "Consider adding a type alias for this complex generic type to improve readability.", "severity": "Suggestion"},
    {"comment": "You could use destructuring assignment here: const { name, email } = user;", "severity": "Suggestion"},
    {"comment": "Optional chaining (?.) could simplify this null check chain.", "severity": "Suggestion"},
    {"comment": "The function name 'process' is generic. Something like 'validateAndTransform' would be more descriptive.", "severity": "Suggestion"},
    {"comment": "For future extensibility, you might want to use a strategy pattern here instead of switch.", "severity": "Suggestion"},
    {"comment": "Trailing commas in multiline arrays/objects make diffs cleaner. Optional but nice to have.", "severity": "Suggestion"},
    {"comment": "This enum could benefit from explicit string values for better debugging: Status.ACTIVE = 'active'", "severity": "Suggestion"},
    {"comment": "Consider grouping related imports together: stdlib, third-party, then local modules.", "severity": "Suggestion"},
    {"comment": "A brief README in this folder would help newcomers understand the module structure.", "severity": "Suggestion"},
]

# 打印样本量与各级别分布，快速检查是否平衡
print(f"Total reviews in dataset: {len(REVIEWS)}")
print(f"Severity distribution: {Counter(r['severity'] for r in REVIEWS)}")


## 训练 / 验证 / 测试划分（Train / Val / Test Split）

按约 **70% / 15% / 15%** 划分。先固定 `seed` 再 `shuffle`，保证每次划分可复现，便于对比基线与微调。


In [ ]:
# ========== 划分：固定种子打乱，再切成 train / val / test ==========

# 固定随机种子：同一份 REVIEWS 每次划分结果一致（可复现实验）
random.seed(42)
# 浅拷贝后再打乱，避免改动原始 REVIEWS 顺序
shuffled = REVIEWS.copy()
random.shuffle(shuffled)

# 总条数；按比例算 train / val，剩余全部进 test
n = len(shuffled)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

# 切片：[:train] | [train:train+val] | [train+val:]
train_data = shuffled[:n_train]
val_data = shuffled[n_train:n_train + n_val]
test_data = shuffled[n_train + n_val:]

# 打印各集合大小与标签分布（看是否严重偏斜）
print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print(f"\nTrain distribution: {Counter(r['severity'] for r in train_data)}")
print(f"Test distribution: {Counter(r['severity'] for r in test_data)}")


## 零样本基线评估（Zero-Shot Baseline）

先不微调，直接用前沿模型按 system prompt 分类，量出「要被 QLoRA 打败」的基线准确率。


In [ ]:
# ========== 基线：用 system prompt 做零样本严重级别分类 ==========

# 发给模型的分类说明：保留英文（影响行为的 prompt 不翻译）
SYSTEM_PROMPT = """You are a code review assistant that classifies the severity of code review comments.
Classify each comment into exactly one severity level:

- Critical: Security vulnerability, crash risk, data loss, authentication bypass
- Major: Bug, broken functionality, performance issue, missing error handling
- Minor: Code smell, maintainability concern, missing docs, dead code
- Suggestion: Style preference, optional improvement, naming convention

Respond with ONLY the severity level (Critical, Major, Minor, or Suggestion), nothing else."""

def predict_baseline(comment: str) -> str:
    """用 OpenRouter/OpenAI 做零样本分类；尽量把自由文本映射回 SEVERITIES 之一。"""
    try:
        # Chat Completions：system 定规则，user 放待分类评论
        response = client.chat.completions.create(
            model=FRONTIER_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Classify this code review comment:\n\n{comment}"}
            ],
            # 只要级别词，限制 token 降低跑偏与费用
            max_tokens=10,
            # temperature=0：尽量确定性输出，便于评估可比
            temperature=0
        )
        if response.choices:
            # 取出回复并去空白；模型偶发多写解释，下面用子串匹配纠偏
            raw = (response.choices[0].message.content or "").strip()
            for level in SEVERITIES:
                # 大小写不敏感：只要回复里出现级别名就采纳
                if level.lower() in raw.lower():
                    return level
            # 匹配失败时退回原文或默认 Minor
            return raw or "Minor"
        return "Minor"
    except Exception as e:
        # API 失败时打印错误并给保守默认，避免整条评估流水线中断
        print(f"Prediction error: {e}")
        return "Minor"

# 抽一条测试集样本做冒烟检查（smoke test）
test_review = test_data[0]
prediction = predict_baseline(test_review["comment"])
print(f"Sample comment: {test_review['comment'][:80]}...")
print(f"True: {test_review['severity']} | Predicted: {prediction}")


In [ ]:
# ========== 评估：在数据集上算准确率（accuracy）==========

def calculate_accuracy(predictor, data, verbose=False):
    """对 data 逐条调用 predictor，统计预测==标签的比例，并收集明细。"""
    # 正确条数计数器
    correct = 0
    # 每条样本的结果明细（截断 comment，方便打印）
    results = []
    
    for item in data:
        # 用传入的预测函数（这里是 predict_baseline）打标签
        pred = predictor(item["comment"])
        # 是否与金标一致
        is_correct = pred == item["severity"]
        if is_correct:
            correct += 1
        results.append({
            "comment": item["comment"][:60] + "...",
            "true": item["severity"],
            "predicted": pred,
            "correct": is_correct
        })
        if verbose:
            # 逐条打印 OK / WRONG，便于肉眼看错分模式
            status = "OK" if is_correct else "WRONG"
            print(f"[{status}] True: {item['severity']:10} | Pred: {pred:10} | {item['comment'][:50]}...")
    
    # 空集保护：避免除零
    accuracy = correct / len(data) if data else 0.0
    return accuracy, results

# 在测试集上跑基线，verbose=True 打印每一条
print("Evaluating baseline model on test set...\n")
baseline_accuracy, baseline_results = calculate_accuracy(predict_baseline, test_data, verbose=True)
print(f"\n=== Baseline Accuracy: {baseline_accuracy:.1%} ===")
print("This is the number to beat with QLoRA fine-tuning!")


## 为 QLoRA 准备训练数据

把每条样本打成基座模型（如 Llama 3）的 **chat template** 文本：system → user → assistant（金标 severity）。


In [ ]:
# ========== 训练格式：Llama 风格特殊 token 包一层对话 ==========

# 训练时用的短 system：要求只回一个级别词（英文 prompt 保持原样）
TRAIN_SYSTEM_MSG = """Classify the severity of code review comments.
Respond with exactly one word: Critical, Major, Minor, or Suggestion."""

def format_for_training(item):
    """把一条 {comment, severity} 打成 QLoRA SFT 用的 text 字段。"""
    return {
        # text：整段对话；assistant 侧写金标，供因果语言模型学习
        "text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{TRAIN_SYSTEM_MSG}<|eot_id|><|start_header_id|>user<|end_header_id|>

Classify this code review comment:
{item['comment']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{item['severity']}<|eot_id|>""",
        "comment": item["comment"],
        "severity": item["severity"]
    }

# 预览一条格式化样本，确认特殊 token 与标签位置正确
sample = format_for_training(train_data[0])
print("=== Formatted Training Example ===")
print(sample["text"])


In [ ]:
# ========== 批量格式化：train / val / test 全部转成训练样本 ==========

# 列表推导：对每个划分调用 format_for_training
train_formatted = [format_for_training(item) for item in train_data]
val_formatted = [format_for_training(item) for item in val_data]
test_formatted = [format_for_training(item) for item in test_data]

print(f"Formatted: {len(train_formatted)} train, {len(val_formatted)} val, {len(test_formatted)} test")


## 导出数据集到 HuggingFace Hub

先写本地 JSONL；需要在 Google Colab 训练时，再取消注释把 `DatasetDict` 推到 Hub。


In [ ]:
# ========== 本地导出：每行一个 JSON（JSONL），方便 datasets 加载 ==========

# 输出目录名；不存在就创建
JSONL_DIR = "jsonl"
os.makedirs(JSONL_DIR, exist_ok=True)

def write_jsonl(data, filepath):
    # 文本模式写入；确保每行 dumps 一个对象
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

# 三个划分各写一个文件
write_jsonl(train_formatted, f"{JSONL_DIR}/train.jsonl")
write_jsonl(val_formatted, f"{JSONL_DIR}/validation.jsonl")
write_jsonl(test_formatted, f"{JSONL_DIR}/test.jsonl")

print(f"Exported to {JSONL_DIR}/")
print(f"  - train.jsonl: {len(train_formatted)} examples")
print(f"  - validation.jsonl: {len(val_formatted)} examples")
print(f"  - test.jsonl: {len(test_formatted)} examples")


In [ ]:
# ========== 可选：推送到 HuggingFace Hub（默认整块注释，避免误推）==========

# 若要在 Colab 用 load_dataset(DATASET_NAME)，取消下方三引号注释后运行
# 需要环境变量 HF_TOKEN，并把 your-username 改成你的 Hub 用户名

'''
from datasets import Dataset, DatasetDict
from huggingface_hub import login

# Login to HuggingFace
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)
    
    # Create dataset
    dataset = DatasetDict({
        "train": Dataset.from_list(train_formatted),
        "validation": Dataset.from_list(val_formatted),
        "test": Dataset.from_list(test_formatted)
    })
    
    # Push to Hub (change username)
    HF_USERNAME = "your-username"  # Change this!
    DATASET_NAME = f"{HF_USERNAME}/code-review-severity"
    dataset.push_to_hub(DATASET_NAME, private=True)
    print(f"Dataset pushed to: https://huggingface.co/datasets/{DATASET_NAME}")
else:
    print("HF_TOKEN not found. Set it to push to HuggingFace Hub.")
'''


## （可选）用 LLM 扩充训练数据

数据少时，可让前沿模型按同样四级定义再生成一批合成评审；默认调用代码注释掉，避免意外花额度。


In [ ]:
# ========== 数据增强：让前沿模型按级别批量生成合成评审 ==========

def generate_reviews(n_per_severity: int = 5):
    """为每个严重级别生成 n_per_severity 条真实感评审评论（JSON 数组）。"""
    # 生成 prompt 保持英文：要求只回 JSON，便于 json.loads
    prompt = f"""Generate {n_per_severity} realistic code review comments for each severity level.

Severity definitions:
- Critical: Security vulnerability, crash risk, data loss, authentication bypass
- Major: Bug, broken functionality, performance issue, missing error handling  
- Minor: Code smell, maintainability concern, missing docs, dead code
- Suggestion: Style preference, optional improvement, naming convention

Make them diverse across different programming languages and scenarios.

Reply with ONLY a JSON array, no other text:
[{{"comment": "review comment text", "severity": "Critical"}}, ...]"""
    
    try:
        # 调用前沿模型；温度略高以增加样本多样性
        response = client.chat.completions.create(
            model=FRONTIER_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=3000,
            temperature=0.8  # 稍高温度：希望样本多样
        )
        if response.choices:
            raw = (response.choices[0].message.content or "").strip()
            # 去掉可能的 ```json ... ``` 围栏，只留纯 JSON
            raw = re.sub(r"^```(?:json)?\s*", "", raw).strip()
            raw = re.sub(r"\s*```$", "", raw).strip()
            
            generated = json.loads(raw)
            # 校验：必须是 dict，severity 落在 SEVERITIES，且有 comment
            valid = [
                g for g in generated 
                if isinstance(g, dict) and g.get("severity") in SEVERITIES and g.get("comment")
            ]
            return valid
    except Exception as e:
        print(f"Generation failed: {e}")
    return []

# 需要扩充时取消下面三行注释：
# new_reviews = generate_reviews(5)
# print(f"Generated {len(new_reviews)} new reviews")
# REVIEWS.extend(new_reviews)


## Gradio UI：交互式代码评审分类器

把基线分类、测试集评估、导出说明和数据统计放进同一个界面，便于演示「微调前」的效果。


In [ ]:
# ========== Gradio 回调：分类展示 / 评估报告 / 抽样 / 导出说明 ==========

# 各级别在 UI 上的颜色（十六进制）；用于结果高亮
SEVERITY_COLORS = {
    "Critical": "#dc3545",   # 红：必须立刻修
    "Major": "#fd7e14",      # 橙：应尽快修
    "Minor": "#ffc107",      # 黄：质量改进
    "Suggestion": "#28a745"  # 绿：可选建议
}

# 给用户看的行动建议文案（UI 展示字符串，保持英文与原逻辑一致）
SEVERITY_DESCRIPTIONS = {
    "Critical": "Must fix before merge - security/crash risk",
    "Major": "Should fix - bug or significant issue",
    "Minor": "Nice to fix - code quality improvement",
    "Suggestion": "Optional - style or preference"
}

def classify_review(comment: str) -> str:
    """Gradio：对单条评审跑基线分类，返回带颜色的 Markdown。"""
    if not comment.strip():
        return "Please enter a code review comment."
    
    # 复用零样本预测函数
    severity = predict_baseline(comment)
    color = SEVERITY_COLORS.get(severity, "#6c757d")
    desc = SEVERITY_DESCRIPTIONS.get(severity, "")
    
    # 返回 Markdown（含 HTML span 上色）；文案保持英文以匹配原 UI
    return f"""## Classification Result

**Severity:** <span style="color:{color}; font-weight:bold; font-size:1.3em;">{severity}</span>

**Action:** {desc}

---

### Severity Guide:
- **Critical**: Security vulnerabilities, crashes, data loss
- **Major**: Bugs, broken functionality, performance issues
- **Minor**: Code smells, maintainability, missing docs
- **Suggestion**: Style preferences, optional improvements"""

def run_evaluation() -> str:
    """在测试集上跑完整评估，拼出 Markdown 报告（含分级别准确率）。"""
    accuracy, results = calculate_accuracy(predict_baseline, test_data)
    
    output = f"""## Evaluation Results

**Model:** {FRONTIER_MODEL}  
**Test Set Size:** {len(test_data)}  
**Accuracy:** {accuracy:.1%}

### Sample Predictions:

| Status | True | Predicted | Comment |
|--------|------|-----------|--------|
"""
    
    # 只展示前 10 条，避免 UI 过长
    for r in results[:10]:
        status = "OK" if r["correct"] else "WRONG"
        output += f"| {status} | {r['true']} | {r['predicted']} | {r['comment'][:40]}... |\n"
    
    # 按真实标签聚合：看哪一类最容易分错
    by_severity = {}
    for r in results:
        key = r["true"]
        if key not in by_severity:
            by_severity[key] = {"correct": 0, "total": 0}
        by_severity[key]["total"] += 1
        if r["correct"]:
            by_severity[key]["correct"] += 1
    
    output += "\n### Accuracy by Severity:\n\n"
    for sev in SEVERITIES:
        if sev in by_severity:
            stats = by_severity[sev]
            pct = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
            output += f"- **{sev}**: {stats['correct']}/{stats['total']} ({pct:.0f}%)\n"
    
    return output

def get_sample_review(severity: str) -> str:
    """从 REVIEWS 里按级别随机抽一条 comment，填进输入框做演示。"""
    matching = [r for r in REVIEWS if r["severity"] == severity]
    if matching:
        return random.choice(matching)["comment"]
    return ""

def show_export_info() -> str:
    """展示 JSONL 路径与 QLoRA 微调步骤说明（Markdown）。"""
    return f"""## Training Data Export

| File | Examples |
|------|----------|
| {JSONL_DIR}/train.jsonl | {len(train_formatted)} |
| {JSONL_DIR}/validation.jsonl | {len(val_formatted)} |
| {JSONL_DIR}/test.jsonl | {len(test_formatted)} |

### QLoRA Fine-Tuning Steps:

1. **Upload to HuggingFace Hub** (or use JSONL directly)
2. **Open Google Colab** with T4 GPU runtime
3. **Load base model** (e.g., Llama 3.2 3B) with 4-bit quantization
4. **Apply LoRA adapters** to attention layers
5. **Train** for 2-3 epochs with learning rate ~2e-4
6. **Evaluate** and compare to baseline

### Example Colab Setup:

```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05
)
```
"""


In [ ]:
# ========== 组装 Gradio Blocks：分类 / 评估 / 训练说明 / 数据统计四个 Tab ==========

# Soft 主题；title 出现在浏览器标签
with gr.Blocks(title="Code Review Classifier", theme=gr.themes.Soft()) as demo:
    # 顶栏说明：UI 展示文案保持英文（与原界面一致）
    gr.Markdown("""# Code Review Severity Classifier
    
Classify code review comments into severity levels using LLM.
This demonstrates the baseline before QLoRA fine-tuning.""")
    
    with gr.Tabs():
        # Tab 1：单条分类
        with gr.Tab("Classify Review"):
            with gr.Row():
                with gr.Column():
                    review_input = gr.Textbox(
                        label="Code Review Comment",
                        placeholder="Enter a code review comment to classify...",
                        lines=4
                    )
                    with gr.Row():
                        classify_btn = gr.Button("Classify", variant="primary")
                        clear_btn = gr.Button("Clear")
                    
                    gr.Markdown("### Load Sample Review:")
                    with gr.Row():
                        # 每个严重级别一个按钮；lambda 默认参数绑定当前 sev，避免闭包踩坑
                        for sev in SEVERITIES:
                            btn = gr.Button(sev, size="sm")
                            btn.click(
                                fn=lambda s=sev: get_sample_review(s),
                                outputs=review_input
                            )
                
                with gr.Column():
                    result_output = gr.Markdown(label="Result")
            
            # 绑定分类与清空
            classify_btn.click(fn=classify_review, inputs=review_input, outputs=result_output)
            clear_btn.click(fn=lambda: ("", ""), outputs=[review_input, result_output])
        
        # Tab 2：测试集评估
        with gr.Tab("Evaluate Model"):
            gr.Markdown("""### Baseline Evaluation
            
Run the zero-shot classifier on the test set to measure baseline accuracy.
This is the number to beat with QLoRA fine-tuning!""")
            
            eval_btn = gr.Button("Run Evaluation", variant="primary")
            eval_output = gr.Markdown()
            
            eval_btn.click(fn=run_evaluation, outputs=eval_output)
        
        # Tab 3：QLoRA / 导出说明
        with gr.Tab("QLoRA Training"):
            gr.Markdown("""### Fine-Tuning Instructions
            
Export training data and follow QLoRA fine-tuning steps.""")
            
            export_btn = gr.Button("Show Export Info", variant="primary")
            export_output = gr.Markdown()
            
            export_btn.click(fn=show_export_info, outputs=export_output)
        
        # Tab 4：数据集统计（构建时就算好 Markdown）
        with gr.Tab("Dataset Info"):
            dataset_info = f"""### Dataset Statistics

| Split | Count |
|-------|-------|
| Total | {len(REVIEWS)} |
| Train | {len(train_data)} |
| Validation | {len(val_data)} |
| Test | {len(test_data)} |

### Severity Distribution:

| Severity | Count | Percentage |
|----------|-------|------------|
"""
            counts = Counter(r['severity'] for r in REVIEWS)
            for sev in SEVERITIES:
                count = counts.get(sev, 0)
                pct = count / len(REVIEWS) * 100
                dataset_info += f"| {sev} | {count} | {pct:.1f}% |\n"
            
            dataset_info += f"""\n### Model Configuration

- **Baseline Model:** {FRONTIER_MODEL}
- **Recommended Base Model:** Llama 3.2 3B or Qwen 2.5 3B
- **QLoRA Rank:** 16
- **Target Modules:** q_proj, v_proj, k_proj, o_proj
"""
            gr.Markdown(dataset_info)

# 启动 Gradio；在笔记本里会弹出本地链接
demo.launch()


## QLoRA 训练代码（给 Google Colab 用）

下面单元格把完整训练脚本放进字符串 `COLAB_CODE` 并打印。请复制到 **带 T4 GPU** 的 Colab 笔记本运行（本机无 GPU 时不要直接跑那段安装/训练逻辑）。


In [ ]:
# ========== Colab 训练脚本：放在字符串里，复制到 T4 运行时执行 ==========

# 说明：这是「给 Colab 复制」的模板字符串，不是在本机立刻训练
# 内含：装依赖 → 登录 HF → 4bit 量化加载 → LoRA → SFTTrainer → push_to_hub

COLAB_CODE = '''
# Install dependencies
!pip install -q transformers peft bitsandbytes accelerate datasets huggingface_hub trl

import os
import torch
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# Login to HuggingFace
login(userdata.get("HF_TOKEN"), add_to_git_credential=True)

# Configuration
BASE_MODEL = "meta-llama/Llama-3.2-3B"  # or "Qwen/Qwen2.5-3B"
HF_USERNAME = "your-username"  # Change this!
DATASET_NAME = f"{HF_USERNAME}/code-review-severity"
OUTPUT_MODEL = f"{HF_USERNAME}/code-review-classifier"

# Load dataset
dataset = load_dataset(DATASET_NAME)
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])}")

# 4-bit quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    fp16=True,
    report_to="none",
)

# Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
)

# Train!
trainer.train()

# Save to HuggingFace Hub
model.push_to_hub(OUTPUT_MODEL, private=True)
tokenizer.push_to_hub(OUTPUT_MODEL, private=True)
print(f"Model saved to: https://huggingface.co/{OUTPUT_MODEL}")
'''

# 打印整段脚本，方便一键复制到 Colab
print("=== Copy this code to Google Colab with T4 GPU ===")
print(COLAB_CODE)


## 小结

本练习串起了第 7 周主线：

1. **数据集**：合成代码评审评论 + 严重级别标签  
2. **基线评估**：前沿模型零样本准确率（要被打败的数字）  
3. **训练数据**：打成 QLoRA / chat template 可用的 `text`  
4. **导出**：本地 JSONL +（可选）HuggingFace Hub，供 Colab 训练  
5. **Gradio UI**：交互分类与评估界面  

### 预期量级（大致）

| 模型 | 预期准确率 |
|------|------------|
| 随机基线 | ~25% |
| GPT-4.1-mini（零样本） | ~60–75% |
| QLoRA 微调 Llama 3B | ~80–90% |

### 下一步

1. 把数据集推到 HuggingFace Hub  
2. 在 Google Colab（T4 GPU）跑 QLoRA 训练  
3. 拿微调模型准确率对比基线  
4. 部署微调模型做离线推理  
